In [ ]:
import os
import ast
import json
import numpy as np
from utils.logger import setup_logger
import utils.vector_db as vector_db

import logging
import Levenshtein
import re
from tqdm import tqdm_notebook as tqdm
from collections import defaultdict, Counter

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import jieba
from openai import OpenAI


from pathlib import Path
from tqdm import tqdm_notebook as tqdm

from sentence_transformers import SentenceTransformer

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import hashlib
import matplotlib.pyplot as plt

In [ ]:
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
DEFAULT_MODEL = "meta-llama/llama-3.1-8b-instruct"
DATA_PATH = "./data/新型电力系统信息物理安全防护体系研究/"
OUTPUT_PATH = "./output/test_data"

if not os.path.exists(DATA_PATH): exit(0)
if not os.path.exists(OUTPUT_PATH): os.makedirs(OUTPUT_PATH)

In [ ]:
# logging.basicConfig(level=logging.INFO)
# for handler in logging.root.handlers[:]:
#     logging.root.removeHandler(handler)

# log_file_path = f'{OUTPUT_PATH}/LKD-KGC.log'
# logger = setup_logger('logger', log_file_path, overwrite=True)
# 1. 配置路径
OUTPUT_PATH = './output' # 确保这个变量定义了
log_path = f'{OUTPUT_PATH}/LKD-KGC.log'

# 2. 初始化 logger (overwrite=True 表示每次运行清空旧日志)
logger = setup_logger('LKD_Logger', log_path, overwrite=True)

# 3. 在之后的函数里使用
# 注意：不需要在函数里重新定义 logger，直接引用全局变量，
# 或者在函数里调用 logger = logging.getLogger('LKD_Logger') 也能获取到同一个实例。

logger.info("程序开始运行...")

In [ ]:
embed_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device="cuda:0")

In [ ]:
# from functools import wraps
# import time
# from openai import Timeout
# import requests

# #解决限流问题
# def traffic_limit(max_qpm, max_qps):
#     def decorator(func):
#         qps_stack = []
#         qpm_stack = []

#         @wraps(func)
#         def wrapper(*args, **kwargs):
#             now = time.time()
#             while qps_stack and now - qps_stack[0] > 1:
#                 qps_stack.pop(0)
#             while qpm_stack and now - qpm_stack[0] > 60:
#                 qpm_stack.pop(0)        
                
#             # 检查当前调用是否超过了限制
#             if len(qps_stack) >= max_qps:
#                 print("waiting for QPS control")
#                 time.sleep(1.1)                
#             if len(qpm_stack) >= max_qpm:
#                 try:
#                     sleep_time = qpm_stack[0]+60-time.time()+5
#                     print(f"waiting for QPM Control: {sleep_time}s")
#                     time.sleep(sleep_time)
#                 except: time.sleep(60)
                
#             qps_stack.append(time.time())
#             qpm_stack.append(time.time())
#             return func(*args, **kwargs)
#         return wrapper
#     return decorator

# #rewrite this function to access your own LLM
# @traffic_limit(10, 1)
# def request_model(prompt, context=None, model=DEFAULT_MODEL, temperature=0.1, max_tokens=30000):
#     prompt = prompt[:max_tokens]
#     msgs = [{"role":"system",   "content": "Please strictly follow the user’s required output format, and answer in English"},
#         {"role": "user", "content": prompt}]
#     client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    
#     if context is not None:
#         context += msgs
#         msgs = context
    
#     extend_fields= {"top_k": 1, "max_new_tokens":1000}
#     try:
#         response = client.chat.completions.create(
#             model=model,
#             messages=msgs,
#             stream=False,
#             temperature=temperature,
#             max_tokens=max_tokens,
#             timeout=Timeout(120, 20),
#             top_p=0.8,
#             extend_fields=extend_fields)      

#         if context is not None: context.append({"role":"assistant", "content": response.choices[0].message.content})

#         if response.error_code is None:
#             resp_text = json.dumps(response.choices[0].message.content, ensure_ascii=False)
#             think_pattern = r'<think>.*?</think>'
#             clean_text = re.sub(think_pattern, '', resp_text)
#             return clean_text
#         else:
#             logger.error( f'request fail: {response.error_code}')
#             raise
            
#     except: 
#         return None

# print(request_model("hello"))
from functools import wraps
import time
import os
import json
import re
import traceback
from openai import OpenAI, Timeout 

# 这里不需要再重新定义 logger = logging.getLogger(__name__) 
# 因为上面我们配置的是 Root Logger，直接用 logging.info/error 即可，或者用全局 logger 变量



def traffic_limit(max_qpm, max_qps):
    def decorator(func):
        qps_stack = []
        qpm_stack = []

        @wraps(func)
        def wrapper(*args, **kwargs):
            now = time.time()
            while qps_stack and now - qps_stack[0] > 1:
                qps_stack.pop(0)
            while qpm_stack and now - qpm_stack[0] > 60:
                qpm_stack.pop(0)        
                
            if len(qps_stack) >= max_qps:
                # [修改点] print -> logger.warning
                logger.warning("Triggered QPS control: waiting...")
                time.sleep(1.1)                
            if len(qpm_stack) >= max_qpm:
                try:
                    sleep_time = qpm_stack[0]+60-time.time()+5
                    # [修改点] print -> logger.warning
                    logger.warning(f"Triggered QPM Control: sleeping for {sleep_time:.2f}s")
                    time.sleep(sleep_time)
                except: 
                    time.sleep(60)
                
            qps_stack.append(time.time())
            qpm_stack.append(time.time())
            return func(*args, **kwargs)
        return wrapper
    return decorator

@traffic_limit(10, 1)
def request_model(prompt, context=None, model=DEFAULT_MODEL, temperature=0.1, max_tokens=2000):
    prompt = prompt[:max_tokens]
    msgs = [{"role":"system", "content": "Please strictly follow the user’s required output format, and answer in Chinese"},
            {"role": "user", "content": prompt}]
    
    try:
        # 建议：如果是批量跑，Client 最好在外层初始化，不要放在函数里
        client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'), base_url="https://openrouter.ai/api/v1")
    except Exception as e:
        # [修改点] print -> logger.error
        logger.error(f"Client Init Error: {e}")
        return None
    
    if context is not None:
        msgs = context + msgs 
    
    # extra_body 逻辑...
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=msgs,
            stream=False,
            temperature=temperature,
            max_tokens=max_tokens,
            timeout=120, 
            top_p=0.8,
        )      

        content = response.choices[0].message.content
        
        if context is not None: 
            context.append({"role":"assistant", "content": content})

        think_pattern = r'<think>.*?</think>'
        clean_text = re.sub(think_pattern, '', content, flags=re.DOTALL)
        
        # [可选] 记录一条成功日志，方便调试
        # logger.info(f"Success request for prompt: {prompt[:20]}...")
        
        return clean_text.strip()

    except Exception as e:
        # [修改点] print -> logger.error
        # 使用 exc_info=True 可以自动把堆栈信息写入日志文件，代替 traceback.print_exc()
        logger.error(f"Request Fail Error: {e}", exc_info=True)
        return None

if __name__ == "__main__":
    if 'OPENAI_API_KEY' not in os.environ:
        logger.error("请设置 OPENAI_API_KEY 环境变量")
    else:
        logger.info("开始测试模型请求...")
        res = request_model("hello")
        logger.info(f"测试结果: {res}")

In [ ]:
###util functions
def lsdir(dir_path):
    dir_subs = os.listdir(dir_path)
    return [item for item in dir_subs if not item.startswith('.')]

def count_file_num(KB_dir):
    total_files = 0
    for dirpath, dirnames, filenames in os.walk(KB_dir):
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        total_files += len(filenames)
    return total_files

def check_key_existence(KB_dir, json_dir):
    json_file = load_dict_from_file(json_dir)
    for dirpath, dirnames, filenames in os.walk(KB_dir):
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        for fname in filenames:
            file_path = os.path.join(dirpath, fname)
            if os.path.abspath(file_path) not in json_file.keys(): print(file_path)
    return

def find_best_match(a_element, b_list):
    best_match = None
    best_similarity = -1  
    for b_element in b_list:
        distance = Levenshtein.distance(a_element, b_element)
        similarity = 1 - (distance / max(len(a_element), len(b_element)))
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_match = b_element
            
    return best_match

def save_dict_to_file(data_dict, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_dict, f, ensure_ascii=False, indent=4)

def load_dict_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)


def parse_model_resp(resp_type):
    def decorator(func):
        def wrapper(*args, **kwargs):
            resp = func(*args, **kwargs)
            
            def extract_formatted_data(text):
                try:
                    sections = text.split('###')
                except: raise
                    
                if len(sections) < 2:
                    raise ZeroDivisionError
                
                formatted_text = sections[1].replace('\\n', '').replace('\r', '')
                formatted_text = formatted_text.replace("\\", "")
                formatted_text = formatted_text.replace("“", "\"").replace("”", "\"").replace("：",":").replace("，", ",").replace("；", ";").replace(";\"", "\"")
                token_count = len(word_tokenize(formatted_text))
                if token_count > 3000:  return ""
                #print(formatted_text)
                if resp_type == "list":
                    try:
                        def ensure_list_quotes(input_str):
                            input_str = input_str.strip()[1:-1]
                            elements = [element.strip() for element in input_str.split(',')]
                            
                            quoted_elements = [
                                element if (element.startswith('"') and element.endswith('"')) or \
                                           (element.startswith("'") and element.endswith("'"))
                                else f'"{element}"'
                                for element in elements
                            ]
                            

                            result = '[' + ','.join(quoted_elements) + ']'
                            return result
                        result = ast.literal_eval(ensure_list_quotes(formatted_text))
                        return result
                    except (SyntaxError, ValueError) as e:
                        logger.error("Error parsing model response:", e, formatted_text)
                        raise
                        
                elif resp_type == "str":
                    return formatted_text.replace("\"","").replace("\'","")
                    
                elif resp_type == "original_str":
                    return formatted_text
                    
                elif resp_type == "dict":
                    try:
                        #print(formatted_text)
                        #formatted_text =  re.sub(r'^"(.*)"$', r'\'\1\'', formatted_text)
                        result = ast.literal_eval(formatted_text)
                        return result
                    except Exception as e:
                        logger.error("Error parsing model response:", e, formatted_text)
                        raise

            # 调用内部函数并返回结果
            parsed_response = extract_formatted_data(resp)
            logger.info(parsed_response)
            return parsed_response

        return wrapper

    return decorator

In [ ]:
#bottom-up summary

# @parse_model_resp(resp_type="original_str")
# def summarize_no_ref(path, history):
#     if os.path.isfile(path):
#         title = os.path.basename(path)
#         with open(path, "r") as f:
#             doc_content = f.read()
#             #doc_content = "\n".join(line for line in doc_content.splitlines() if line.strip() != "")
    
#         prompt = f"""
#             ----------------Task Requirements-------------------
            
#             Briefly summarize the content of the input document in one sentence. Answer in English.
            
#             ----------------Current Input Document-------------------
            
#             Title: {title}
            
#             Content: {doc_content}
            
#             ---------------Output Format Requirements--------------------
            
#             Return a string marked with ### at the beginning and end, and the string must not contain single or double quotes. The format is as follows:
#             ###
#             Summary content
#             ###
#         """
#     elif os.path.isdir(path):
#         dirname =  os.path.basename(path)
#         sub_summaries = ""
#         for entry in os.listdir(path):
#             if entry.startswith(".") or entry.startswith("__"): continue
#             entry_path = os.path.abspath(os.path.join(path, entry))
#             sub_summaries += f"""
#                 Name: {entry}
#                 Content: {history[entry_path]}
                
#             """
#         prompt = f"""
#             ----------------Task Requirements-------------------
            
#             Given a directory called {dirname} and summaries of its documents or subdirectories, briefly summarize a one-sentence summary of the directory's contents.
            
#             ----------------Summaries of documents/subdirectories------------------
            
#             {sub_summaries}
            
#             ---------------Output Format Requirements--------------------
            
#             Return a string marked with ### at the beginning and end, and the string must not contain single or double quotes. The format is as follows:
#             ###
#             Summary content
#             ###
#         """
#     logger.info(prompt)
#     resp = request_model(prompt)
#     print(path, resp)
#     return resp

# 1. 删除或注释掉这个装饰器，它太脆弱了
# @parse_model_resp(resp_type="original_str") 
def summarize_no_ref(path, history):
    if os.path.isfile(path):
        title = os.path.basename(path)
        # 建议加上 encoding='utf-8' 防止Windows下读取中文报错
        with open(path, "r", encoding='utf-8') as f:
            doc_content = f.read()
    
        prompt = f"""
            ----------------Task Requirements-------------------
            Briefly summarize the content of the input document in one sentence. Answer in Chinese.
            ----------------Current Input Document-------------------
            Title: {title}
            Content: {doc_content[:15000]} 
            ---------------Output Format Requirements--------------------
            Return a string marked with ### at the beginning and end. 
            ###
            Summary content
            ###
        """
    elif os.path.isdir(path):
        dirname =  os.path.basename(path)
        sub_summaries = ""
        for entry in os.listdir(path):
            if entry.startswith(".") or entry.startswith("__"): continue
            entry_path = os.path.abspath(os.path.join(path, entry))
            # 使用 .get 防止 history 中缺少 key 导致报错
            content = history.get(entry_path, "")
            sub_summaries += f"""
                Name: {entry}
                Content: {content}
            """
        prompt = f"""
            ----------------Task Requirements-------------------
            Given a directory called {dirname} and summaries of its documents or subdirectories, briefly summarize a one-sentence summary of the directory's contents.
            ----------------Summaries of documents/subdirectories------------------
            {sub_summaries}
            ---------------Output Format Requirements--------------------
            Return a string marked with ### at the beginning and end.
            ###
            Summary content
            ###
        """
    
    # 2. 直接调用模型
    logger.info("Sending request...")
    logger.info(prompt)
    resp = request_model(prompt)
    print(path, resp)
    
    # 3. 【关键修改】手动且宽容的解析逻辑
    if resp:
        clean_resp = resp.strip()
        # 如果模型听话加了 ###，我们就取中间
        if "###" in clean_resp:
            parts = clean_resp.split("###")
            if len(parts) >= 3:
                # 正常情况：### 内容 ###
                final_summary = parts[1].strip()
            elif len(parts) == 2:
                # 只有开头或结尾有一个 ###
                final_summary = parts[1].strip() if parts[1].strip() else parts[0].strip()
            else:
                final_summary = clean_resp.replace("###", "").strip()
        else:
            # 如果模型没加 ###，直接用它的回答，不要报错！
            final_summary = clean_resp
            
        # 简单的清洗
        final_summary = final_summary.replace("\n", " ").strip('"').strip("'")
        print(path, final_summary)
        return final_summary
    else:
        return ""
    
#post-order traverse directory tree
def summarize_dir_bottom_up(root_dir):
    summaries = {}
    root_dir = os.path.abspath(root_dir)
    for entry in os.listdir(root_dir):
        if entry.startswith(".") or entry.startswith("__"): continue
        entry_path = os.path.join(root_dir, entry)
        if os.path.isfile(entry_path):
            summaries[entry_path] = summarize_no_ref(entry_path, summaries)
        elif os.path.isdir(entry_path):
            summaries.update(summarize_dir_bottom_up(entry_path))
    summaries[root_dir] = summarize_no_ref(root_dir, summaries)
    return summaries
    

bottom_up_summaries = summarize_dir_bottom_up(DATA_PATH)

In [ ]:
# # @parse_model_resp(resp_type="list")
# # def sort_dir(dir_path, bootm_up_summaries, context=None): 
# #     sub_summaries = ""
# #     for entry in os.listdir(dir_path):
# #         if entry.startswith(".") or entry.startswith("__"): continue
# #         entry_path = os.path.abspath(os.path.join(dir_path, entry))
# #         sub_summaries += f"""
# #             Name: {entry}
# #             Content: {bottom_up_summaries[entry_path]}
            
# #         """
# #     prompt = f"""
# #     ----------------Task Requirements-------------------
# #     Here is a dicectory called <{os.path.basename(dir_path)}>, which descripts: <{bottom_up_summaries[os.path.abspath(dir_path)]}>.
# #     It has some directories or documents, along with their summaries.
# #     If you need to read these documents in sequence to build a knowledge graph, and when processing subsequent documents, you can refer to the previous ones, what order do you think would best utilize the existing knowledge?
# #     Note: If there is only one document, return the original document as is. If there are multiple documents, you should prioritize reading summary or overview documents first, followed by documents that provide background knowledge, and finally documents that involve specific contents.
# #     If there is historical dialogue context, you need to consider why the previous responses in the conversation were incorrect and then answer according to the requirements.
    
# #     ----------------Summaries of documents/subdirectories------------------

# #     {sub_summaries}
    
# #     ---------------Output Format Requirements--------------------
# #     Provide your reasoning process and return the following format with priorities sorted from highest to lowest, marked by ###:
# #     Note only the list should be marked by ###, but not the reasoning process.
# #     ###
# #     [Returned priority list]
# #     ###
# #     Example output format:
# #     ###['A','B']###
# #     """
# #     logger.info(prompt)
# #     resp =  request_model(prompt, context)
# #     #print(resp)
# #     return resp

# # import re
# # import ast
# # 1. 移除装饰器，避免因缺少 ### 而报错
# # @parse_model_resp(resp_type="list") 
# def sort_dir(dir_path, bottom_up_summaries, context=None): 
#     sub_summaries = ""
#     # 获取真实的文件列表，用于后续对比或Fallback
#     real_files = [f for f in os.listdir(dir_path) if not (f.startswith(".") or f.startswith("__"))]
    
#     for entry in real_files:
#         entry_path = os.path.abspath(os.path.join(dir_path, entry))
#         # 使用 .get 防止 Key Error
#         summary = bottom_up_summaries.get(entry_path, "No summary available")
#         sub_summaries += f"""
#             Name: {entry}
#             Content: {summary}
            
#         """
        
#     prompt = f"""
#     ----------------Task Requirements-------------------
#     Here is a directory called <{os.path.basename(dir_path)}>.
#     It has some directories or documents, along with their summaries.
#     If you need to read these documents in sequence to build a knowledge graph, what order do you think would best utilize the existing knowledge?
    
#     Prioritize reading summary/overview documents first, followed by background knowledge, and finally specific contents.
    
#     ----------------Summaries of documents/subdirectories------------------

#     {sub_summaries}
    
#     ---------------Output Format Requirements--------------------
#     Return a Python list of strings marked with ###. 
#     Example: ###['intro.md', 'concept.md']###
#     Only return the list, no reasoning.
#     """
    
#     logger.info(f"Sorting directory: {dir_path}")
#     logger.info(prompt)
    
#     # 2. 获取原始响应
#     try:
#         resp = request_model(prompt, context)
#     except Exception as e:
#         logger.error(f"Model request failed: {e}")
#         return real_files # 失败时返回默认顺序，保证程序不崩
        
#     if not resp:
#         return real_files

#     # 3. 【核心修改】鲁棒的列表提取逻辑
#     try:
#         # 步骤A: 尝试提取 ### 中间的内容，如果没有 ###，就用全文
#         if "###" in resp:
#             content_str = resp.split("###")[1].strip()
#         else:
#             content_str = resp.strip()
            
#         # 步骤B: 使用正则表达式寻找列表格式 ['...', '...']
#         # dotall=True 让 . 可以匹配换行符
#         match = re.search(r'\[.*\]', content_str, re.DOTALL)
        
#         if match:
#             list_str = match.group()
#             # 步骤C: 安全地将字符串转为 Python List
#             sorted_list = ast.literal_eval(list_str)
            
#             # 简单验证一下是不是列表
#             if isinstance(sorted_list, list):
#                 logger.info(f"Parsed sorted list: {sorted_list}")
#                 return sorted_list
#             else:
#                 logger.warning("Parsed result is not a list, returning default.")
#                 return real_files
#         else:
#             logger.warning("No list format found in response, returning default.")
#             return real_files
            
#     except Exception as e:
#         logger.error(f"Error parsing sort result: {e}. Resp: {resp}")
#         # 如果解析崩了，千万别抛错，直接返回原始列表，让流程继续走下去！
#         return real_files


# def refine_subnames(base, dir_subs):
#     true_subnames = os.listdir(base)
#     def replace_with_best_matches(A, B):
#         replaced_list = [find_best_match(a, B) for a in A]
#         return replaced_list
        
#     return replace_with_best_matches(dir_subs, true_subnames)


# #ref_sums: {path: sum, }
# def summarize_with_ref(doc_path, vecDB_path, max_context=20):
#     title = os.path.basename(doc_path)
#     with open(doc_path, "r") as f:
#         doc_content = f.read()
#         #doc_content = "\n".join(line for line in doc_content.splitlines() if line.strip() != "")
    
#     if not os.path.exists(vecDB_path): ref_sums=[]
#     else: ref_sums = vector_db.query(doc_content, max_context, embed_model, vecDB_path)
        
#     ref_context = ""
#     for i, ref in enumerate(ref_sums):
#         #print(ref_path)
#         ref_context += f"{i+1}. {ref} \n"
    
#     prompt = f"""
#     ----------------Task Requirements-------------------
    
#     Briefly summarize the content of the current input document, answering in the same language as the input text.

#     ----------------Current Input Document-------------------
    
#     Title: {title}
    
#     Content: {doc_content}
        
#     ---------------Output Format Requirements--------------------
    
#     Return a string that summarizes the content of the input document, without single or double quotes.
    
#     ---------------Background Knowledge-------------------
#     The input document is located in a subdirectory of the following document, and all entities mentioned within the document are within the context of the background knowledge. The background knowledge may include domain introductions, terminology definitions, operational methods, etc. Please consider this background knowledge when summarizing the input document. Note that you should summarize the current input document, referencing the background knowledge, rather than treating the background knowledge as the main document.
#     Background knowledge:
#     {ref_context}
#     """
#     logger.info(prompt)
#     return request_model(prompt)


# def summarize_dir_up_down(KB_dir, bottom_up_summaries, save_dir, max_context=20):
#     total_files = 0
#     accessed_files = 0
#     for dirpath, dirnames, filenames in os.walk(KB_dir):
#         dirnames[:] = [d for d in dirnames if not d.startswith('.')]
#         total_files += len(filenames)
    
#     docs_summary = {}
#     vecDB_path = os.path.join(os.path.dirname(save_dir), "summary_faiss_index.bin")
#     if os.path.exists(vecDB_path): os.remove(vecDB_path)
#     vecDB_dir = os.path.dirname(vecDB_path)
#     if not os.path.exists(vecDB_dir):
#         os.makedirs(vecDB_dir)
        
#     def do_summarize(base_dir, filetype=".md"):
#         nonlocal accessed_files
#         def equal_to_list(ele, lst):
#             if not isinstance(ele, list):
#                 return False
#             return all(ele.count(x) == lst.count(x) for x in set(lst))
            
#         if lsdir(base_dir) == []: return 
#         logger.info("\n-------------------")
#         logger.info(f"base dir: {base_dir}")
#         sort_context = []
#         dir_subs = sort_dir(base_dir, bottom_up_summaries, sort_context) 
        
#         max_retry = 5
#         while not(equal_to_list(dir_subs, lsdir(base_dir))) and max_retry>0:
#             logger.error(f"invalid sorted directory: {dir_subs}, retry...")
#             dir_subs = sort_dir(base_dir, bottom_up_summaries, sort_context)
#             logger.info(f"newly sorted dir: {dir_subs}")     
#             dir_subs = refine_subnames(base_dir, dir_subs)
#             logger.info(f"refined sorted dir: {dir_subs}")
#             max_retry -= 1

#         if not(equal_to_list(dir_subs, lsdir(base_dir))): 
#             logger.error(f"sorting failed: {base_dir}")
#             return
    
#         logger.info(f"final sorted dir: {dir_subs}")  
#         for sub in dir_subs:
#             abs_path = os.path.abspath(os.path.join(base_dir, sub))
#             print(abs_path)
#             if not os.path.exists(abs_path):
#                 logger.error(f"invalid path：{abs_path}")
#                 continue
                
#             if os.path.isfile(abs_path) and abs_path.endswith(filetype):
#                 try:
#                     logger.info(abs_path)
#                     logger.info(f"ref_docs: { [os.path.basename(docname) for docname in docs_summary.keys()] }")
#                     single_doc_summary = summarize_with_ref(abs_path, vecDB_path, max_context)
#                     print(single_doc_summary)
#                     docs_summary[abs_path] = single_doc_summary
#                     vector_db.index_text(single_doc_summary, f"title：{os.path.basename(abs_path)}， content : {single_doc_summary}", embed_model, vecDB_path)
#                     accessed_files += 1
#                     print(f"{accessed_files}/{total_files}")
#                 except:
#                     logger.error(f"error processing {abs_path}")
#                     print(f"error processing {abs_path}")
#                     continue
#             elif os.path.isdir(abs_path):
#                 do_summarize(abs_path)  
#         return
        
#     do_summarize(KB_dir)
#     save_dict_to_file(docs_summary, save_dir)
#     #os.remove(vecDB_path)
#     print("done")
#     return docs_summary
    
# summarize_dir_up_down(DATA_PATH, bottom_up_summaries, f"{OUTPUT_PATH}/docs_summary.json", 10)


import os
import logging
import ast
import re
from difflib import SequenceMatcher

# 假设 logger, request_model, vector_db, embed_model, save_dict_to_file, DATA_PATH, OUTPUT_PATH 已定义

def lsdir(path):
    """辅助函数：获取目录下非隐藏文件列表"""
    if not os.path.exists(path): return []
    return [f for f in os.listdir(path) if not (f.startswith(".") or f.startswith("__"))]

# ================= 新增：强制对齐函数 =================
def align_file_order(real_files, llm_sorted_names):
    """
    根据 LLM 返回的排序建议 (llm_sorted_names) 对 真实文件 (real_files) 进行排序。
    保证：
    1. 返回的列表包含且仅包含 real_files 中的所有文件。
    2. 尽可能遵循 llm_sorted_names 的顺序。
    """
    if not llm_sorted_names:
        return real_files

    # 创建一个未处理文件的集合
    pending_files = list(real_files)
    ordered_files = []

    # 辅助函数：模糊匹配
    def find_best_match(target, candidates):
        best_match = None
        best_ratio = 0.0
        # 1. 尝试完全匹配
        if target in candidates:
            return target
        # 2. 尝试去后缀匹配 (比如 LLM 返回 'Intro'，真实文件 'Intro.md')
        for cand in candidates:
            if cand == target: 
                return cand
            if os.path.splitext(cand)[0] == target: # 匹配文件名不带后缀
                return cand
            
            # 3. 相似度匹配 (防止少许错别字)
            ratio = SequenceMatcher(None, target, cand).ratio()
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = cand
        
        # 阈值：如果相似度太低（比如 < 0.4），则认为没找到
        if best_ratio > 0.4:
            return best_match
        return None

    # 遍历 LLM 的建议
    for name in llm_sorted_names:
        match = find_best_match(name, pending_files)
        if match:
            ordered_files.append(match)
            pending_files.remove(match) # 移除已处理的文件

    # 将剩余没被 LLM 提到的文件追加到末尾
    if pending_files:
        logger.info(f"Files missed by LLM sorting, appending to end: {pending_files}")
        ordered_files.extend(pending_files)

    return ordered_files

# ================= 修改后的 sort_dir =================
def sort_dir(dir_path, bottom_up_summaries, context=None): 
    # 获取真实文件列表作为 fallback
    real_files = lsdir(dir_path)
    if not real_files: return []
    if len(real_files) == 1: return real_files # 只有一个文件不需要排序

    sub_summaries = ""
    for entry in real_files:
        entry_path = os.path.abspath(os.path.join(dir_path, entry))
        # 使用 .get 避免 KeyError
        summary = bottom_up_summaries.get(entry_path, "No summary provided.")
        # 截断一下摘要，防止 token 溢出
        sub_summaries += f"""
            Name: {entry}
            Content: {summary[:500]} 
        """
        
    prompt = f"""
    ----------------Task Requirements-------------------
    Here is a directory called <{os.path.basename(dir_path)}>.
    It contains the following documents/subdirectories: {real_files}
    
    Summaries:
    {sub_summaries}
    
    If you need to read these documents in sequence to build a knowledge graph, what order do you think would best utilize the existing knowledge?
    Prioritize reading summary/overview documents first, followed by background knowledge, and finally specific contents.
    
    ---------------Output Format Requirements--------------------
    Return a Python list of strings marked with ###. 
    The strings must be EXACTLY the filenames provided in the input list (including extensions like .md).
    Example: ###['intro.md', 'concept.md']###
    Only return the list.
    """
    
    logger.info(f"Sorting directory: {dir_path}")
    
    try:
        resp = request_model(prompt, context)
        
        # 解析逻辑
        clean_resp = resp.strip()
        if "###" in clean_resp:
            clean_resp = clean_resp.split("###")[1].strip()
        
        # 尝试提取列表部分
        match = re.search(r'\[.*\]', clean_resp, re.DOTALL)
        if match:
            list_str = match.group()
            sorted_list = ast.literal_eval(list_str)
            if isinstance(sorted_list, list):
                logger.info(f"LLM suggested order: {sorted_list}")
                return sorted_list
            
    except Exception as e:
        logger.error(f"Error in sort_dir: {e}. Returning default order.")
    
    return real_files

# ================= 摘要生成函数 (保持原样，略微优化) =================
def summarize_with_ref(doc_path, vecDB_path, max_context=20):
    title = os.path.basename(doc_path)
    # 增加 encoding 防止 Windows 报错
    with open(doc_path, "r", encoding='utf-8') as f:
        doc_content = f.read()

    if not os.path.exists(vecDB_path): 
        ref_sums = []
    else: 
        # 增加 try-except 防止向量库查询偶尔报错
        try:
            ref_sums = vector_db.query(doc_content[:1000], max_context, embed_model, vecDB_path)
        except:
            ref_sums = []
        
    ref_context = ""
    for i, ref in enumerate(ref_sums):
        ref_context += f"{i+1}. {ref} \n"
    
    prompt = f"""
    ----------------Task Requirements-------------------
    Briefly summarize the content of the current input document in Chinese.
    ----------------Current Input Document-------------------
    Title: {title}
    Content: {doc_content[:15000]}
    ---------------Output Format Requirements--------------------
    Return a string summary. No quotes.
    ---------------Background Knowledge-------------------
    {ref_context}
    """
    # logger.info(prompt) # 减少日志刷屏
    return request_model(prompt)

# ================= 核心逻辑：递归摘要 =================
# def summarize_dir_up_down(KB_dir, bottom_up_summaries, save_dir, max_context=20):
#     total_files = 0
#     accessed_files = 0
#     # 统计文件总数
#     for dirpath, dirnames, filenames in os.walk(KB_dir):
#         dirnames[:] = [d for d in dirnames if not d.startswith('.')]
#         total_files += len(filenames)
    
#     docs_summary = {}
#     vecDB_path = os.path.join(os.path.dirname(save_dir), "summary_faiss_index.bin")
    
#     # 清理旧的向量库
#     if os.path.exists(vecDB_path): 
#         try: os.remove(vecDB_path)
#         except: pass
        
#     vecDB_dir = os.path.dirname(vecDB_path)
#     if not os.path.exists(vecDB_dir): os.makedirs(vecDB_dir)
        
#     def do_summarize(base_dir, filetype=".md"):
#         nonlocal accessed_files
        
#         real_files = lsdir(base_dir)
#         if not real_files: return 
        
#         logger.info("\n-------------------")
#         logger.info(f"Processing dir: {base_dir}")
        
#         # 1. 获取 LLM 建议的顺序
#         # 注意：这里我们不需要 context 了，因为我们不打算重试
#         llm_suggested_order = sort_dir(base_dir, bottom_up_summaries, None)
        
#         # 2. 【关键】强制对齐，不再报错，不再重试
#         # 使用 align_file_order 保证结果一定是 real_files 的一个排列
#         final_sorted_dir = align_file_order(real_files, llm_suggested_order)
        
#         logger.info(f"Final execution order: {final_sorted_dir}")
        
#         # 3. 按顺序执行
#         for sub in final_sorted_dir:
#             abs_path = os.path.abspath(os.path.join(base_dir, sub))
            
#             if not os.path.exists(abs_path):
#                 logger.error(f"Path not found (skipped): {abs_path}")
#                 continue
                
#             if os.path.isfile(abs_path) and abs_path.endswith(filetype):
#                 try:
#                     logger.info(f"Summarizing file: {sub}")
#                     single_doc_summary = summarize_with_ref(abs_path, vecDB_path, max_context)
                    
#                     # 简单的清洗
#                     if single_doc_summary:
#                         single_doc_summary = single_doc_summary.strip().replace('"', '').replace("'", "")
                    
#                     docs_summary[abs_path] = single_doc_summary
                    
#                     # 存入向量库
#                     vector_db.index_text(single_doc_summary, f"title：{sub}， content : {single_doc_summary}", embed_model, vecDB_path)
                    
#                     accessed_files += 1
#                     print(f"Progress: {accessed_files}/{total_files}")
                    
#                 except Exception as e:
#                     logger.error(f"Error processing file {abs_path}: {e}")
#                     continue
                    
#             elif os.path.isdir(abs_path):
#                 # 递归处理子目录
#                 do_summarize(abs_path)  
#         return
        
#     # 开始执行
#     do_summarize(KB_dir)
#     save_dict_to_file(docs_summary, save_dir)
#     print("Summarization process completed.")
#     return docs_summary
def summarize_dir_up_down(KB_dir, bottom_up_summaries, save_dir, max_context=20):
    KB_dir = os.path.abspath(KB_dir) # 强制转换为绝对路径
    logger.info(f"Root Directory for processing: {KB_dir}")

    total_files = 0
    # 统计总文件数
    for dirpath, dirnames, filenames in os.walk(KB_dir):
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        total_files += len(filenames)
    
    logger.info(f"Total files found in directory tree: {total_files}")
    if total_files == 0:
        logger.error("No files found! Please check DATA_PATH.")
        return {}

    docs_summary = {}
    vecDB_path = os.path.join(os.path.dirname(save_dir), "summary_faiss_index.bin")
    
    # 清理旧的向量库
    if os.path.exists(vecDB_path): 
        try: os.remove(vecDB_path)
        except: pass
        
    vecDB_dir = os.path.dirname(vecDB_path)
    if not os.path.exists(vecDB_dir): os.makedirs(vecDB_dir)
        
    accessed_files = 0

    def do_summarize(base_dir):
        nonlocal accessed_files
        
        real_files = lsdir(base_dir)
        if not real_files: 
            logger.info(f"Skipping empty directory: {base_dir}")
            return 
        
        logger.info(f"Enter directory: {base_dir}")
        
        # 1. 获取 LLM 建议顺序
        llm_suggested_order = sort_dir(base_dir, bottom_up_summaries, None)
        
        # 2. 强制对齐
        final_sorted_dir = align_file_order(real_files, llm_suggested_order)
        logger.info(f"Order determined: {final_sorted_dir}")
        
        # 3. 遍历文件
        for sub in final_sorted_dir:
            abs_path = os.path.abspath(os.path.join(base_dir, sub))
            
            # Debug: 打印当前检查的路径
            # print(f"Checking: {abs_path}") 

            if os.path.isfile(abs_path):
                # 【修改点1】移除严格的 .md 后缀限制，或者增加 .txt
                # 如果你想限制，改成: if abs_path.endswith((".md", ".txt")):
                if not abs_path.startswith("."): 
                    try:
                        logger.info(f"Summarizing file: {sub}")
                        
                        # 调用摘要生成
                        single_doc_summary = summarize_with_ref(abs_path, vecDB_path, max_context)
                        
                        # 【修改点2】检查返回值
                        if not single_doc_summary:
                            logger.warning(f"Warning: Empty summary returned for {sub}")
                            # 即使为空也写入，方便排查
                            single_doc_summary = "Summary generation failed."

                        # 清洗
                        single_doc_summary = single_doc_summary.strip().replace('"', '').replace("'", "")
                        
                        docs_summary[abs_path] = single_doc_summary
                        
                        # 存入向量库
                        vector_db.index_text(single_doc_summary, f"title：{sub}， content : {single_doc_summary}", embed_model, vecDB_path)
                        
                        accessed_files += 1
                        print(f"Progress: {accessed_files}/{total_files} | Processed: {sub}")
                        
                    except Exception as e:
                        # 【修改点3】详细打印报错堆栈
                        import traceback
                        logger.error(f"CRITICAL ERROR processing {sub}: {e}")
                        traceback.print_exc() # 打印完整报错信息
                        continue
                else:
                    logger.info(f"Ignored hidden file: {sub}")

            elif os.path.isdir(abs_path):
                do_summarize(abs_path)
        return
        
    # 开始执行
    do_summarize(KB_dir)
    
    # 检查结果
    if not docs_summary:
        logger.error("!!! docs_summary is EMPTY. No files were successfully processed. !!!")
    else:
        logger.info(f"Saving summaries for {len(docs_summary)} files.")
        save_dict_to_file(docs_summary, save_dir)
        print("Summarization process completed.")
        
    return docs_summary

# 运行
summarize_dir_up_down(DATA_PATH, bottom_up_summaries, f"{OUTPUT_PATH}/docs_summary.json", 10)

In [ ]:
# #tags: {tag:weight}(weighted) or [tag,](not weighted)
# def kmeans_clust(tags_with_weight, weighted=False, visualize=False):
#     def find_origin_tag(embed, tags, embeddings):
#         cosine_similarities = cosine_similarity([embed], embeddings)[0] 
#         #print(embed.shape, embeddings.shape, cosine_similarities.shape)
#         most_similar_index = np.argmax(cosine_similarities)
#         origin_tag = tags[most_similar_index]
#         return origin_tag
        
#     if weighted:
#         tags = list(tags_with_weight.keys())
#         #weights = log_norm(list(tags_with_weight.values()))
#         weights = list(tags_with_weight.values())
#         #print(weights)
        
#     else:
#         tags = tags_with_weight
#         weights = None
        
#     embeddings = embed_model.encode(tags)
    
#     silhouette_scores = []
#     distinct_tags_num = len(set(tags))
#     max_clusters = min(100, distinct_tags_num)  

#     for n_clusters in range(2, max_clusters - 1):
#         kmeans = KMeans(n_clusters=n_clusters, random_state=42)
#         kmeans.fit(embeddings, sample_weight=weights)
        
#         if len(set(kmeans.labels_)) > 1:
#             score = silhouette_score(embeddings, kmeans.labels_)
#             silhouette_scores.append(score)
#         else:
#             silhouette_scores.append(-1)  
#     #print(silhouette_scores)
#     best_n_clusters = np.argmax(silhouette_scores) + 2 
#     best_score = silhouette_scores[best_n_clusters - 2]
    
#     print(f"best cluster number: {best_n_clusters} (Silhouette Coefficient: {best_score:.3f})")

#     #best_n_clusters = 4
#     kmeans_best = KMeans(n_clusters=best_n_clusters, random_state=42)
#     y_kmeans = kmeans_best.fit_predict(embeddings, sample_weight=weights)
    
#     cluster_centers = kmeans_best.cluster_centers_

#     clusters_info = []
    
#     for i in range(best_n_clusters):
#         indices = np.where(kmeans_best.labels_ == i)[0]
        
#         cluster_vectors = embeddings[indices]
#         cluster_tags = []
#         for vec in cluster_vectors:
#             cluster_tags.append(find_origin_tag(vec, tags, embeddings))
        
#         center_vector = kmeans_best.cluster_centers_[i]
#         center_tag = find_origin_tag(center_vector, tags, embeddings)
        
#         clusters_info.append({
#             'vectors': cluster_vectors,
#             "all_tags": cluster_tags,
#             'center_vector': center_vector,
#             "center_tag": center_tag
#         })

#     if visualize:
#         pca = PCA(n_components=2) 
#         embeddings_pca = pca.fit_transform(embeddings)
        
#         plt.figure(figsize=(10, 6))
        
#         scatter = plt.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1], 
#                               c=y_kmeans, s=100, cmap='viridis', alpha=0.7, label='Cluster Labels')
        
#         centers_pca = pca.transform(kmeans.cluster_centers_)
#         #plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', s=200, alpha=0.75, marker='X', label='Cluster Centers')
        
#         cbar = plt.colorbar(scatter)
#         cbar.set_label('Cluster Label')
        
#         plt.title('KMeans Clustering Visualization of Embeddings')
#         plt.xlabel('Principal Component 1')
#         plt.ylabel('Principal Component 2')
#         plt.legend()
#         plt.grid()
#         plt.show()
    
#     return clusters_info

# def filter_tags(input_list, min_times=2):
#     #input_list = [lemmatizer.lemmatize(word.lower(), pos='n')  for word in input_list]
#     count = Counter(input_list)
#     merged_list = [item for item in count if count[item] >= min_times]
#     print(f"tags number:{len(merged_list)}")
#     return merged_list

# def split_tags(merged_list, group_size=10):
#     return [merged_list[i:i + group_size] for i in range(0, len(merged_list), group_size)]
    
# #docs_sum: {doc_path:summary, }
# #return: [entity_tag, ]
# def extract_entity_tags(doc_path, docs_sum):
#     doc_path = os.path.abspath(doc_path)
#     doc_summary = docs_sum[doc_path]
#     with open(doc_path, "r") as f:
#         doc_content = f.read()
#         #doc_content = "\n".join(line for line in doc_content.splitlines() if line.strip() != "")
#     doc_title = os.path.basename(doc_path)
#     doc_info = (doc_title, doc_summary, doc_content)
#     context  = []

#     @parse_model_resp(resp_type="list")
#     def parse_list(prompt, context):
#         return request_model(prompt, context)

#     @parse_model_resp(resp_type="str")
#     def parse_str(prompt, context):
#         return request_model(prompt, context)
        
        
#     entity_extraction_prompt = f"""
#         ----------------Task Requirements-------------------
#         Next, a text and its summary will be provided. You need to refer to the summary and the content of the text to classify meaningful entities within the text. Note that you only need to return the type names of the summary, not the specific entity names, and the number of entity types should be kept to a minimum. Each entity type should be in singular form and begin with an uppercase letter.
        
#         ---------------Output Format Requirements--------------------
#         Return a list containing all the class names in the following format, with the list being wrapped in ### markers and each type being a string marked with single quotes, formatted as follows:
#         ###
#         [Return type list]
#         ###
#         ---------------Current Input--------------------
#         Current input text summary:
#         《{doc_info[0]}》 : {doc_info[1]}
        
#         Current input text content:
#         {doc_info[2]}
#         """
    
#     logger.info(entity_extraction_prompt)
    
#     max_try = 10
#     first_try = True 
#     while max_try>0:
#         try:
#             if first_try:
#                 first_try = False 
#                 entity_types = parse_list(entity_extraction_prompt, context)
#             else: entity_types = parse_list(format_error_prompt, context)
#             assert isinstance(entity_types, list)
#             return list(set(entity_types))
#         except Exception as err:
#             err_msg = f"err: {err}"
#             logger.error(err_msg)
#             format_error_prompt = f"""
#                 The returned object does not meet the format requirements, an issue was encountered while parsing the return content: {err_msg}.
                
#                 Please check and correct the previous return result. The requirements are as follows:
                
#                 The return format must be a list object marked with ### at both the beginning and end, where each element is a string marked with single quotes, and no quotes are allowed inside the element strings.
#                 The entity types should be meaningful and concise.
#                 Continue to modify the format of the returned object so that it satisfies the above format requirements, i.e., a list containing all the class names marked with ### at both the beginning and end.
                
#                 Note that you only need to provide the corrected result, without the process or code.               
#                 """
#             print(f"error: {err_msg}, retry")
#             max_try -= 1 

#     print("exceed max retry, return []")
#     return []
            

# #return: {doc_path:[entity_type, ], }
# def extract_KB_entity_tags(KB_dir, docs_sum):
#     KB_schema = {}
    
#     total_files = 0
#     accessed_files = 0
#     for dirpath, dirnames, filenames in os.walk(KB_dir):
#         dirnames[:] = [d for d in dirnames if not d.startswith('.')]
#         total_files += len(filenames)
        
#     for root, dirs, files in os.walk(KB_dir):
#         dirs[:] = [d for d in dirs if not d.startswith('.')]
#         for file in files:
#             try:
#                 doc_path = os.path.abspath(os.path.join(root, file))
#                 print(doc_path)
#                 doc_entity_tags = extract_entity_tags(doc_path, docs_sum)
#                 KB_schema[doc_path] = doc_entity_tags
#                 accessed_files += 1
#                 print(f"{accessed_files}/{total_files}")
#             except Exception as err:
#                 logger.error(f"error processing {doc_path}")
#                 print(err)
#                 continue
#     return KB_schema

# #tag_schema: json {path:[tag, ], }
# #return:[{tag:definition， }, ]
# def comment_KB_entity_tag(tag_schema, ref_summary_path, ref_num=20, min_times=2):
#     @parse_model_resp(resp_type="str")
#     def ask_comment(prompt):
#         return request_model(prompt)

#     @parse_model_resp(resp_type="dict")
#     def refine_tags(context):
#         refine_prompt = """
#             Check and correct the previous return result. The requirements are as follows:
            
#             Pay attention to the background knowledge and consider the meanings represented by various types in the context of the provided background knowledge to improve the specificity of the descriptions and avoid being overly broad.
#             Merge entities of the same meaning or with a clear inclusion relationship. For entities A and B that have a clear inclusion relationship, where A includes B, merge them into the broader type A. For example, "server" and "http server" should be merged into "server."
#             The return format must be a JSON object marked with ### at the beginning and end, where both keys and values are enclosed in double quotes. Each key is a type name, and the value is the definition of that type. However, there should be no extra quotes inside the key and value strings, or around the JSON object itself.
#             """
#         return request_model(refine_prompt, context)
        
#     full_entity_schema = []    
#     tags = []
#     processed_clusters = 0
#     for doc in tag_schema:
#         entity_tags = tag_schema[doc]
#         for tag in entity_tags:
#             tags.append(tag)

#     if count_file_num(DATA_PATH)>50: min_times=5
#     tags = filter_tags(tags, min_times)
#     print(tags)

#     clusters_info = kmeans_clust(tags, False, False)
#     for cluster in clusters_info:
#         print(f"{processed_clusters+1}/{len(clusters_info)}")
#         processed_clusters += 1
#         cluster_tag_groups = split_tags(cluster["all_tags"], 3)
#         print(cluster_tag_groups)
#         for cluster_tags in cluster_tag_groups:
#             ref_sums = vector_db.query(str(cluster_tags), ref_num, embed_model, ref_summary_path)
#             ref_context = ""
#             for i, ref in enumerate(ref_sums):
#                 #print(ref_path)
#                 ref_context += f"{i+1}. {ref} \n"
    
            
#             extraction_prompt = f"""
#                 ---------------------------------Task Requirements----------------------------------
#                 There is a series of entity types. First, correct any spelling errors, then replace types that have the same meaning or have a clear inclusion relationship with one common term, and briefly define each merged type in one sentence.
#                 When defining, make sure to incorporate background knowledge as much as possible to improve the specificity of the descriptions and avoid being overly broad.
#                 For entity types A and B that have a clear inclusion relationship where A includes B, merge them into the broader type A. For example, "server" and "http server" should be merged into "server."
#                 The input entity types are as follows: {cluster_tags}
                
#                 -----------------------------------Output Format Requirements------------------------------------
                
#                 Return a JSON object containing all merged class definitions in the following format, where both the key and the value are marked with single quotes. Special attention is needed: the key and value strings must not contain any quotes internally. The JSON object should be marked with ### at the beginning and end, and should follow the format below, but keep in mind to only reference the format of the example, and do not treat the example as input:
#                 ###
#                 {{"entity_type": "description and definition", }}
#                 ###

#                 -----------------------------------Example input------------------------------------
#                 ['Receiver', 'Header', 'Message', 'Series', 'Protocol'] 

#                 -----------------------------------Example Output------------------------------------
#                 ###{{'Receiver': 'A component in the Prometheus ecosystem that accepts and processes incoming metrics or alerts, often part of a monitoring or alerting system.', 'Header': 'A metadata section in a data packet or message that contains control information, used to manage the transmission and processing of metrics or alerts in Prometheus.', 'Message': 'A unit of communication in Prometheus that contains metrics, alerts, or other data, transmitted between components such as exporters, servers, and AlertManager.', 'Series': 'A sequence of timestamped data points in Prometheus, representing a time series used for monitoring and analysis, often associated with a metric and labeled dimensions.', 'Protocol': 'A set of rules and conventions in Prometheus that govern the format and transmission of data, such as the exposition format or remote-write protocol, ensuring reliable and efficient communication between components.'}}###
                
#                 -----------------------------------Background Knowledge-----------------------------------
                
#                 These entity types are part of the same system, for which there is documentation that may include domain introductions, terminology definitions, operational methods, and other content. Please merge and briefly define and describe each type using all available documentation:
#                 {ref_context}
#             """
        
#             #print(cluster_tags, "\n")
            
#             try:
#                 origin_resp = ask_comment(extraction_prompt)
#                 context = [
#                     {
#                      "role": "user",
#                      "content": extraction_prompt
#                     },
#                     {
#                     "role": "assistant",
#                     "content": origin_resp
#                 }]
#                 refined_resp = refine_tags(context)
#                 full_entity_schema.append(refined_resp)
#                 print(refined_resp)
                
#             except Exception as err:
#                 print(err)
#                 break
#     return full_entity_schema

import os
import logging
import time
import re
import ast  # 用于安全的解析 Python 风格的 list/dict 字符串
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances_argmin_min
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from collections import Counter

# 假设 logger, request_model, vector_db, embed_model, count_file_num, DATA_PATH, OUTPUT_PATH 已经在上下文定义

# ================= 辅助函数：解析模型返回的字符串 =================
def robust_parse_response(response_str, expected_type=list):
    """
    尝试从模型返回的字符串中解析出 list 或 dict。
    兼容 ### 标记、Markdown 代码块、单/双引号混用等情况。
    """
    if not response_str:
        return expected_type()

    clean_str = response_str.strip()

    # 1. 尝试提取 ### 之间的内容
    if "###" in clean_str:
        parts = clean_str.split("###")
        # 取最长的那个部分，或者取中间部分
        if len(parts) >= 3:
            clean_str = parts[1]
        elif len(parts) == 2:
            clean_str = parts[1] if parts[1].strip() else parts[0]
    
    # 2. 尝试提取 Markdown 代码块 (```json ... ``` 或 ``` ... ```)
    match = re.search(r'```(?:\w+)?\s*(.*?)\s*```', clean_str, re.DOTALL)
    if match:
        clean_str = match.group(1)

    clean_str = clean_str.strip()

    # 3. 尝试解析
    try:
        # ast.literal_eval 可以处理 Python 风格的 list/dict (单引号 key/value)
        # 这比 json.loads 更宽容，适合你的 prompt 要求（你 prompt 里要求用单引号）
        parsed = ast.literal_eval(clean_str)
        if isinstance(parsed, expected_type):
            return parsed
    except Exception as e:
        logger.warning(f"Parsing failed with ast: {e}. Raw content: {clean_str[:100]}...")
    
    return expected_type()

# ================= 原有聚类函数 (保持不变) =================
# tags: {tag:weight}(weighted) or [tag,](not weighted)
def kmeans_clust(tags_with_weight, weighted=False, visualize=False):
    def find_origin_tag(embed, tags, embeddings):
        cosine_similarities = cosine_similarity([embed], embeddings)[0] 
        most_similar_index = np.argmax(cosine_similarities)
        origin_tag = tags[most_similar_index]
        return origin_tag
        
    if weighted:
        tags = list(tags_with_weight.keys())
        weights = list(tags_with_weight.values())
    else:
        tags = tags_with_weight
        weights = None
        
    embeddings = embed_model.encode(tags)
    
    silhouette_scores = []
    distinct_tags_num = len(set(tags))
    max_clusters = min(100, distinct_tags_num)  

    # 如果标签太少，直接返回
    if distinct_tags_num < 2:
        return [{
            'vectors': embeddings,
            "all_tags": tags,
            'center_vector': embeddings[0],
            "center_tag": tags[0]
        }]

    for n_clusters in range(2, max_clusters - 1):
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        kmeans.fit(embeddings, sample_weight=weights)
        
        if len(set(kmeans.labels_)) > 1:
            score = silhouette_score(embeddings, kmeans.labels_)
            silhouette_scores.append(score)
        else:
            silhouette_scores.append(-1)  
            
    if not silhouette_scores:
        best_n_clusters = 2
    else:
        best_n_clusters = np.argmax(silhouette_scores) + 2 
        best_score = silhouette_scores[best_n_clusters - 2]
        print(f"best cluster number: {best_n_clusters} (Silhouette Coefficient: {best_score:.3f})")

    kmeans_best = KMeans(n_clusters=best_n_clusters, random_state=42)
    y_kmeans = kmeans_best.fit_predict(embeddings, sample_weight=weights)
    
    clusters_info = []
    
    for i in range(best_n_clusters):
        indices = np.where(kmeans_best.labels_ == i)[0]
        cluster_vectors = embeddings[indices]
        cluster_tags = []
        for vec in cluster_vectors:
            cluster_tags.append(find_origin_tag(vec, tags, embeddings))
        
        center_vector = kmeans_best.cluster_centers_[i]
        center_tag = find_origin_tag(center_vector, tags, embeddings)
        
        clusters_info.append({
            'vectors': cluster_vectors,
            "all_tags": cluster_tags,
            'center_vector': center_vector,
            "center_tag": center_tag
        })

    if visualize:
        # (可视化代码保持不变)
        pass
    
    return clusters_info

def filter_tags(input_list, min_times=2):
    count = Counter(input_list)
    merged_list = [item for item in count if count[item] >= min_times]
    print(f"tags number:{len(merged_list)}")
    return merged_list

def split_tags(merged_list, group_size=10):
    return [merged_list[i:i + group_size] for i in range(0, len(merged_list), group_size)]

# ================= 修改后的提取函数 =================
# docs_sum: {doc_path:summary, }
# return: [entity_tag, ]
def extract_entity_tags(doc_path, docs_sum):
    doc_path = os.path.abspath(doc_path)
    # 增加容错，防止路径不在 summary 中
    doc_summary = docs_sum.get(doc_path, "")
    
    try:
        with open(doc_path, "r", encoding='utf-8') as f: # 加上 encoding 防止乱码
            doc_content = f.read()
    except Exception as e:
        logger.error(f"Cannot read file {doc_path}: {e}")
        return []

    doc_title = os.path.basename(doc_path)
    doc_info = (doc_title, doc_summary, doc_content)
    
    # 移除内部定义的带装饰器函数，直接使用 Prompt
    entity_extraction_prompt = f"""
        ----------------Task Requirements-------------------
        Next, a text and its summary will be provided. You need to refer to the summary and the content of the text to classify meaningful entities within the text. Note that you only need to return the type names of the summary, not the specific entity names, and the number of entity types should be kept to a minimum. Each entity type should be in singular form and begin with an uppercase letter.
        
        ---------------Output Format Requirements--------------------
        Return a list containing all the class names in the following format, with the list being wrapped in ### markers and each type being a string marked with single quotes, formatted as follows:
        ###
        ['Type1', 'Type2', 'Type3']
        ###
        ---------------Current Input--------------------
        Current input text summary:
        《{doc_info[0]}》 : {doc_info[1]}
        
        Current input text content:
        {doc_info[2][:15000]} 
        """
        # 注意：这里限制了 content 长度防止 token 溢出
    
    logger.info(f"Extracting tags for {doc_title}...")
    
    max_try = 3 # 降低重试次数，避免死循环太久
    
    for _ in range(max_try):
        try:
            # 1. 直接请求模型
            resp = request_model(entity_extraction_prompt)
            
            # 2. 使用 robust_parse_response 解析
            entity_types = robust_parse_response(resp, expected_type=list)
            
            if entity_types:
                return list(set(entity_types)) # 去重返回
            else:
                # 如果解析为空，可能需要重试提示
                print(f"Empty or invalid list parsed for {doc_title}, retrying...")
                raise ValueError("Parsed result is not a valid list")

        except Exception as err:
            err_msg = f"err: {err}"
            logger.error(err_msg)
            # 修改 prompt 进行重试 (Simple retry logic)
            entity_extraction_prompt += f"\n\nPrevious response was invalid ({err_msg}). Please STRICTLY follow the format: ### ['TypeA', 'TypeB'] ###"

    print(f"Exceed max retry for {doc_title}, returning []")
    return []

# return: {doc_path:[entity_type, ], }
def extract_KB_entity_tags(KB_dir, docs_sum):
    KB_schema = {}
    
    total_files = 0
    accessed_files = 0
    for dirpath, dirnames, filenames in os.walk(KB_dir):
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        total_files += len(filenames)
        
    for root, dirs, files in os.walk(KB_dir):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for file in files:
            # 过滤非文本文件，视具体情况而定
            if file.startswith('.'): continue 

            try:
                doc_path = os.path.abspath(os.path.join(root, file))
                print(f"Processing: {doc_path}")
                doc_entity_tags = extract_entity_tags(doc_path, docs_sum)
                KB_schema[doc_path] = doc_entity_tags
                accessed_files += 1
                print(f"Progress: {accessed_files}/{total_files}")
            except Exception as err:
                logger.error(f"error processing {doc_path}")
                print(err)
                continue
    return KB_schema

# ================= 修改后的评论/合并函数 =================
# tag_schema: json {path:[tag, ], }
# return:[{tag:definition， }, ]
def comment_KB_entity_tag(tag_schema, ref_summary_path, ref_num=20, min_times=2):
    # 移除装饰器，改用循环内的直接逻辑
    
    full_entity_schema = []    
    tags = []
    processed_clusters = 0
    
    for doc in tag_schema:
        entity_tags = tag_schema[doc]
        for tag in entity_tags:
            tags.append(tag)

    if count_file_num(DATA_PATH) > 50: 
        min_times = 5
    tags = filter_tags(tags, min_times)
    
    if not tags:
        print("No tags found after filtering.")
        return []

    clusters_info = kmeans_clust(tags, False, False)
    
    for cluster in clusters_info:
        print(f"Processing cluster {processed_clusters+1}/{len(clusters_info)}")
        processed_clusters += 1
        
        # 聚类如果只有一个标签，不需要 split
        if len(cluster["all_tags"]) == 0: continue

        cluster_tag_groups = split_tags(cluster["all_tags"], 5) # 稍微减小 group size 提高精准度
        
        for cluster_tags in cluster_tag_groups:
            # 构建 Reference Context
            ref_sums = vector_db.query(str(cluster_tags), ref_num, embed_model, ref_summary_path)
            ref_context = ""
            if ref_sums:
                for i, ref in enumerate(ref_sums):
                    ref_context += f"{i+1}. {ref} \n"
            
            # Prompt 保持逻辑一致，但明确要求 Python Dict 格式以便 ast 解析
            extraction_prompt = f"""
                ---------------------------------Task Requirements----------------------------------
                There is a series of entity types. First, correct any spelling errors, then replace types that have the same meaning or have a clear inclusion relationship with one common term, and briefly define each merged type in one sentence.
                Merge entities of the same meaning. For entities A and B that have a clear inclusion relationship where A includes B, merge them into the broader type A.
                The input entity types are as follows: {cluster_tags}
                
                -----------------------------------Output Format Requirements------------------------------------
                Return a Python Dictionary object containing all merged class definitions.
                Key: Entity type name (String)
                Value: Definition (String)
                Mark the output with ### at the beginning and end.
                Format:
                ###
                {{'EntityA': 'Definition of A', 'EntityB': 'Definition of B'}}
                ###

                -----------------------------------Background Knowledge-----------------------------------
                {ref_context}
            """
            
            try:
                # 1. 初始请求
                origin_resp = request_model(extraction_prompt)
                
                # 2. 尝试解析
                refined_resp = robust_parse_response(origin_resp, expected_type=dict)
                
                # 如果解析失败或者是空的，可以尝试用 Refine Prompt (可选，这里简化为单次请求，因为通常单次 ast 解析就够了)
                # 如果你需要 Refine 逻辑，可以在这里加一个 if not refined_resp: request_model(refine_prompt) ...
                
                if refined_resp:
                    full_entity_schema.append(refined_resp)
                    print(f"Successfully processed group: {list(refined_resp.keys())}")
                else:
                    print("Failed to parse response for cluster group.")

            except Exception as err:
                print(f"Cluster processing error: {err}")
                continue
                
    return full_entity_schema

In [ ]:
time.sleep(3)
docs_sum = load_dict_from_file(f"{OUTPUT_PATH}/docs_summary.json")
entity_tags = extract_KB_entity_tags(DATA_PATH, docs_sum)
save_dict_to_file(entity_tags, f"{OUTPUT_PATH}/entity_tags_bydoc.json")

In [ ]:
entity_tags = load_dict_from_file(f"{OUTPUT_PATH}/entity_tags_bydoc.json")
full_entity_schema = comment_KB_entity_tag(entity_tags, f"{OUTPUT_PATH}/summary_faiss_index.bin", min_times=2)
save_dict_to_file(full_entity_schema, f"{OUTPUT_PATH}/full_entity_schema.json")

In [ ]:
# ###entity extraction

# #return: {tag:[entities]}
# def extract_entity_full_schema(entity_schema, doc_path, docs_sum):
#     with open(doc_path, "r") as f:
#         doc_content = f.read().replace('"', " ").replace('“', " ").replace('”', " ")

#     doc_path = os.path.abspath(doc_path)
#     doc_summary = docs_sum[doc_path].replace('"', " ").replace('“', " ").replace('”', " ")
#     with open(doc_path, "r") as f:
#         doc_content = "\n".join(line for line in doc_content.splitlines() if line.strip() != "")
        
#     doc_title = os.path.basename(doc_path)
#     #doc_info = (doc_title, doc_summary, doc_content)
    
#     @parse_model_resp(resp_type="original_str")
#     def refine_entities_1(prompt, context):
#         return request_model(prompt, context)

#     @parse_model_resp(resp_type="dict")
#     def refine_entities_2(prompt, context):
#         return request_model(prompt, context)

#     extraction_prompt = f"""
#         Here is a document. Please extract meaningful entities from the document based on their types in conjunction with the specific content, and explain the reasons.
        
#         Requirements:
        
#         1.Pay attention to reading the document summary and consider how the summary encapsulates the entities within the document.        
#         2.Entities should be as specific as possible, avoiding being too abstract or broad. And sufficiently rich in number.
#         3.Ensure the extracted entities do exist in the provided text(Detailed Content) but not Entity Type Description. 
#         4.Extract entity All entities should be in singular form and capitalized.
#         5.Return in JSON format, where both keys and values are enclosed in double quotes. Each key is a type name, and the value contains entities of that type, separated by semicolons (;). However, double quotes are not allowed within the key and value strings.
#         6.Ensure that the extracted entities are consistent with the original text. If the original text is Chinese, the entities are expressed in Chinese. If the original text is in English, the entity is in English.
        
#         Enclose the returned JSON object with ### markers at the beginning and end, following the format below. Note that you should only refer to the example format and not consider the example as input:
        
#            ###
#            {{"EntityType1": "Entity1;Entity2;",  }}
#            ###

#         Document Summary:
#         {{
#         {doc_summary}
#         }}  
        
#         Detailed Content:
#         {{
#         {doc_content}
#         }}  
        
#         --------Entity Type Description-------------
#         The following are all possible types of entities and their descriptions. Please only extract entities that can be categorized into the following types. If there are none, return an empty dictionary. Note that you can only extract entities from the above document, not from the entity type descriptions.
#         {{
#         {entity_schema}
#         }}  
#     """
    
#     context = []
    
#     origin_resp = request_model(extraction_prompt, context)

#     refine_prompt_1 = """
#         Check and correct the previous classification results. The requirements are as follows:

#         1.Pay attention to reading the document summary and consider how the summary encapsulates the entities in the document.
#         2.Ensure the extracted entities do exist in the provided text but not from Entity Type Description. Delete the unexisted ones.
#         3.Check if there are any missing entities. If so, add the missing entities to the returned JSON object.
#         4.All entities should be in singular form and capitalized.
#         5.The return format must be a JSON object marked by ### at the beginning and end, with both keys and values marked with double quotes. Each key is a type name, and the value consists of entities under that type, separated by semicolons (;) within the same type and commas (,) between different types. Note that double quotes and parentheses are not allowed within key or value strings. 
#         6.Ensure that the extracted entities are consistent with the original text. If the original text is Chinese, the entities are expressed in Chinese. If the original text is in English, the entity is in English.
#         """

#     refined_resp1 = refine_entities_1(refine_prompt_1, context)
#     print(refined_resp1)

#     format_refine_context = []
#     max_try = 10
#     res = {}
#     while max_try>0:
#         try:
#             doc_entities = ast.literal_eval(refined_resp1)
#             for tag, entities in doc_entities.items():
#                 if entities.strip() == "" : continue
#                 entities_list = (entities.replace("；", ";")).split(";")
#                 entities_list = [item for item in entities_list if (item in doc_content or item.lower() in doc_content) ]
#                 if not entities_list: continue
#                 print(f"{tag}: {entities_list}")
#                 res[tag] = entities_list
#             return res
#         except Exception as err:
#             err_msg = f"err: {err}"
#             logger.error(err_msg)
#             format_error_prompt = f"""
#                 The following object is parsed from the string: {refined_resp1}
#                 The parsing reveals that the object does not meet the formatting requirements. The return format must be a JSON object marked with ### at the beginning and end, where both keys and values are enclosed in double quotes. Each key is a type name, and the value represents the entities under that type, with different entities separated by semicolons (;) and different types separated by commas (,). However, double quotes and parentheses are not allowed within the key and value strings.
#                 Continue to modify the format of the object to meet the aforementioned formatting requirements.
#                 Note that you only need to provide the corrected result, without any process or code.    
#                 """
#             refined_resp1 = refine_entities_1(format_error_prompt, format_refine_context)
#             print(f"error: {err_msg}, retry")
#             max_try -= 1 

#     print("exceed max retry, return {}")
#     return {}

# #entity_schema: [{tag: comment,}, ]
# #return: {doc_path:{tag:[entity,], }, }
# def extract_KB_entities(entity_schema_path, docs_sum_path):
#     docs_sum = load_dict_from_file(docs_sum_path)
#     entity_schema = load_dict_from_file(entity_schema_path)
#     #print(entity_schema)
    
#     doc_num = len(docs_sum)
#     processed_doc = 0
    
#     all_entities = {}

#     for doc_path in docs_sum.keys():
#         all_entities[doc_path] = {}
#         try:
#             doc_entities = extract_entity_full_schema(entity_schema, doc_path, docs_sum)
#             all_entities[doc_path] = doc_entities
#             print(f"{processed_doc+1}/{doc_num}: {doc_path} succeed")
#         except Exception as err:
#             print(f"{processed_doc+1}/{doc_num}: {err} processing {doc_path}")
#             continue
            
#         processed_doc += 1
#         #break
#     return all_entities  
    
import os
import json
import re
import ast
import logging

# 假设 logger, request_model, load_dict_from_file 等已定义

# ================= 1. 强力清洗与解析函数 =================
def robust_parse_json(response_str):
    """
    尝试从模型返回的字符串中提取并解析 JSON 对象。
    支持：
    1. 标准 JSON (json.loads)
    2. Python 字典字符串 (ast.literal_eval)
    3. 自动去除 Markdown 代码块 ```json ... ```
    4. 自动提取 ### ... ### 之间的内容
    """
    if not response_str:
        return {}

    clean_str = response_str.strip()

    # 1. 尝试提取 ### 之间的内容 (优先级最高)
    if "###" in clean_str:
        parts = clean_str.split("###")
        # 找最像 JSON 的那一部分
        for part in parts:
            part = part.strip()
            if part.startswith("{") and part.endswith("}"):
                clean_str = part
                break
        else:
            # 如果没找到大括号包裹的，默认取中间部分
            if len(parts) >= 3:
                clean_str = parts[1].strip()

    # 2. 尝试提取 Markdown 代码块
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', clean_str, re.DOTALL)
    if match:
        clean_str = match.group(1).strip()

    # 3. 常见错误清洗
    # 替换中文顿号、中文分号（如果在引号外，这很难判断，所以简单处理常见情况）
    # 这里主要处理 JSON 结构性的错误，不轻易替换内容里的字符防止破坏实体名
    
    # 4. 尝试 json.loads (标准 JSON)
    try:
        return json.loads(clean_str)
    except:
        pass

    # 5. 尝试 ast.literal_eval (Python 风格字典，支持单引号)
    try:
        return ast.literal_eval(clean_str)
    except:
        pass
        
    # 6. 终极补救：如果是因为最后少了大括号（截断），尝试补全
    if clean_str.startswith("{") and not clean_str.endswith("}"):
        try:
            return json.loads(clean_str + "}")
        except:
            pass

    logger.warning(f"Failed to parse JSON. Raw content preview: {clean_str[:100]}...")
    return {}

# ================= 2. 修复后的实体提取函数 =================
# return: {tag:[entities]}
def extract_entity_full_schema(entity_schema, doc_path, docs_sum):
    # 读取内容并简单清洗引号，防止 JSON 结构破坏
    try:
        with open(doc_path, "r", encoding='utf-8') as f:
            doc_content = f.read()
    except Exception as e:
        logger.error(f"Cannot read file {doc_path}: {e}")
        return {}

    # 获取摘要
    doc_path = os.path.abspath(doc_path)
    doc_summary = docs_sum.get(doc_path, "No summary available.")
    
    # 清洗掉可能破坏 Prompt 结构的字符
    safe_summary = doc_summary.replace('"', "'").replace("\n", " ")
    # 截取 doc_content 防止 token 溢出 (保留前 15000 字符)
    safe_content = doc_content[:15000].replace('"', "'")

    # 移除内部装饰器函数，直接构建 Prompt
    extraction_prompt = f"""
        ----------------Task Requirements-------------------
        Extract meaningful entities from the document based on the provided Entity Types.
        
        Rules:
        1. Only extract entities that strictly belong to the "Entity Type Description" provided below.
        2. Verify that the extracted entities ACTUALLY EXIST in the "Detailed Content".
        3. Output Format: Return a standard JSON object. 
           - Key: Entity Type Name (String)
           - Value: A single String containing entities separated by semicolons (;).
           - Both keys and values must be in DOUBLE QUOTES.
        
        ----------------Format Example-------------------
        ###
        {{
            "Device": "Switch;Router;Firewall", 
            "Location": "Server Room;Beijing"
        }}
        ###
        
        ----------------Input Data-------------------
        [Document Summary]: {safe_summary}
        
        [Entity Type Description]: 
        {entity_schema}
        
        [Detailed Content]: 
        {safe_content}
    """
    
    # 第一次请求：直接提取
    logger.info(f"Extracting entities for: {os.path.basename(doc_path)}")
    origin_resp = request_model(extraction_prompt)
    
    # 解析结果
    extracted_data = robust_parse_json(origin_resp)
    
    # 如果第一次解析失败或为空，尝试简单的 Refine（可选）
    if not extracted_data:
        logger.warning("First extraction failed or empty, retrying with stricter prompt...")
        refine_prompt = extraction_prompt + "\n\nIMPORTANT: The previous response was invalid. Please ensure the output is valid JSON wrapped in ### markers."
        origin_resp = request_model(refine_prompt)
        extracted_data = robust_parse_json(origin_resp)

    # 验证与过滤逻辑
    final_res = {}
    if extracted_data and isinstance(extracted_data, dict):
        for tag, entities_str in extracted_data.items():
            if not isinstance(entities_str, str): 
                continue # 格式不对，跳过
                
            if not entities_str.strip(): 
                continue

            # 处理分隔符：支持中文分号和英文分号
            clean_entities_str = entities_str.replace("；", ";")
            entities_list = clean_entities_str.split(";")
            
            # 验证实体是否在原文中存在 (忽略大小写)
            valid_entities = []
            doc_content_lower = doc_content.lower()
            
            for item in entities_list:
                item = item.strip()
                if not item: continue
                
                # 检查是否存在
                if item in doc_content or item.lower() in doc_content_lower:
                    valid_entities.append(item)
            
            if valid_entities:
                # 去重
                valid_entities = list(set(valid_entities))
                print(f"  -> Found {tag}: {valid_entities}")
                final_res[tag] = valid_entities
    else:
        logger.error(f"Failed to extract entities for {os.path.basename(doc_path)}")

    return final_res

# ================= 3. 批量处理函数 (增加错误处理) =================
# entity_schema: [{tag: comment,}, ]
# return: {doc_path:{tag:[entity,], }, }
def extract_KB_entities(entity_schema_path, docs_sum_path):
    docs_sum = load_dict_from_file(docs_sum_path)
    entity_schema = load_dict_from_file(entity_schema_path)
    
    if not docs_sum:
        print("Error: docs_summary is empty!")
        return {}
        
    doc_num = len(docs_sum)
    processed_doc = 0
    
    all_entities = {}

    for doc_path in docs_sum.keys():
        # 初始化
        all_entities[doc_path] = {}
        
        try:
            print(f"Processing {processed_doc+1}/{doc_num}: {os.path.basename(doc_path)}")
            
            doc_entities = extract_entity_full_schema(entity_schema, doc_path, docs_sum)
            
            all_entities[doc_path] = doc_entities
            
        except Exception as err:
            logger.error(f"CRITICAL ERROR processing {doc_path}: {err}")
            # 打印堆栈以便调试
            import traceback
            traceback.print_exc()
            continue
        finally:
            processed_doc += 1

    return all_entities

In [ ]:
all_entities = extract_KB_entities(f"{OUTPUT_PATH}/full_entity_schema.json", f"{OUTPUT_PATH}/docs_summary.json")
save_dict_to_file(all_entities, f"{OUTPUT_PATH}/all_entities.json")  
len(load_dict_from_file(f"{OUTPUT_PATH}/all_entities.json"))

In [ ]:
# ###triple extraction

# def split_text(content, chunk_size, lang="eng"):
#     # sunfire, prometheus按行分割
#     chunks = content.strip().split(".")
#     chunks = [chunk.strip() for chunk in chunks]
    
#     output_chunks = []
#     current_chunk = []
#     current_tokens = 0

#     for part in chunks:
#         if part=="": continue
#         if lang=="eng":
#             tokens = word_tokenize(part)
#         elif lang=="chn":
#             tokens = jieba.lcut(part) 
#         else: return
#         token_count = len(tokens)
        
#         if current_tokens + token_count <= chunk_size:
#             current_chunk.append(part.strip()) 
#             current_tokens += token_count
#         else:
#             if current_chunk:
#                 output_chunks.append('\n'.join(current_chunk))
#             current_chunk = [part.strip()]
#             current_tokens = token_count

#     if current_chunk:
#         output_chunks.append('\n'.join(current_chunk))
#     return output_chunks

# #entity_list: {tag:[entiies], }
# #return: [entiies, ]
# def filter_entity_by_text(entity_list, text, with_tag=False, entity_schema={}):
#     result = []
#     entity_schema = {k: v for d in entity_schema for k, v in d.items()}

#     for key, value_list in entity_list.items():
#         found_strings = set()
        
#         for string in value_list:
#             if string in text or string.lower() in text:
#                 found_strings.add(string)
        
#         if found_strings:
#             if with_tag:
#                 for entity in found_strings:
#                     try: result.append((entity, key, entity_schema[key]))
#                     except: continue
#             else:
#                 result.extend(list(found_strings))
#     return set(result)


# #entity_schema:[{entity_type: entities}, ]
# #entity_list: (tag:[entiies], )
# #return: [(s:s_type, p, o:o_type), ]
# def extract_triple(entity_schema, entity_list, doc_path, chunk_size=200):
#     entity_tags = list(set(key for item in entity_schema for key in item.keys()))
#     with open(doc_path, "r") as f:
#         doc_content = f.read().replace('"', " ").replace('“', " ").replace('”', " ")

#     doc_path = os.path.abspath(doc_path)

#     @parse_model_resp(resp_type="original_str")
#     def refine_entities(prompt, context, model="default"):
#         if model == "default": return request_model(prompt, context)
#         else: return request_model(prompt, context, model=model)

#     triples = []
#     chunks = split_text(doc_content, chunk_size, lang="eng")
#     for idx,text_chunk in enumerate(chunks):   
#         print(f"chunk: {idx+1} / {len(chunks)}")
#         logger.info(f"{doc_path}, chunk {idx}, \n content: \n{text_chunk}")
#         existing_entities = filter_entity_by_text(entity_list, text_chunk, False)
#         logger.info(f"existing_entities: {existing_entities}")
#         if existing_entities == {}: continue
#         extraction_prompt = f"""
#             A document is as follows:
            
#             {{
#             {text_chunk}
#             }}
            
#             Entities extracted from this document are as follows:
            
#             {{
#             {existing_entities}
#             }}
            
#             Based on the original text, what entities do you believe have clear and meaningful relationships? Provide the relationships between them with the following requirements:
            
#             1.Use list(tuple) format, where each tuple is represented as a triplet (Subject-Relationship-Object). All entities and relationships should be in singular form with the first letter capitalized.
#             2.Both of the subject and object in the tuple must come from the entities provided above.
#             3.The types of relationships should be diversed, and the number of relationship tuples should be maximized.
#             4.The triplets should represent specific, clear, and meaningful relationships and should not include ambiguous relationships like "related to"
#             5.The relationship content (predicate) should be short and clear, expressing the semantic relationship in as few words as possible.
#                     """
        
#         context = []
        
#         origin_resp = request_model(extraction_prompt, context)
#         if origin_resp is None: continue
            
#         format_refine_prompt= f"""
#             For the list object just mentioned, where each element is a tuple representing a relationship, the following conditions need to be checked and corrected, and then return the corrected result:
            
#             1.All entities and relationships in the tuples should be in singular form, and their initial letters should be capitalized.
#             3.The tuple keeps the following format: "Subject Entity", "Relationship", "Object Entity").
#             4.Each element in the tuple should be marked with double quotes " at both ends, and there should be no other quotes. There should be no quotes within the elements or between the tuples; if there are any quotes within an element, remove them.
#             5.Return a list object marked with ### (three # symbols) at both ends, ensuring that the list has exactly one level of square brackets [] and no additional content. The format should be as follows: ###[("Subject Entity", "Relationship", "Object Entity"),]###. Note that the tuples are the content you need to fill in.
#                     """

#         try:
#             refined_resp2 = refine_entities(format_refine_prompt, context)
#         except: continue
            
#         #print(refined_resp2)
#         max_try = 3
#         while max_try>0:
#             try:
#                 refined_resp2 = refined_resp2.replace("）",")").replace("（", "(").replace("：", ":").replace("，", ",")
#                 ret = ast.literal_eval(refined_resp2)
#                 assert isinstance(ret, list)
#                 triples.extend(ret)
#                 print(ret)
#                 break
#             except Exception as err:
#                 max_try -= 1 
#                 err_msg = f"err: {err} at {doc_path} chunk {idx}"
#                 logger.error(err_msg)
#                 format_error_prompt = f"""
#                     From the returned content above, parsed out the following object: {refined_resp2}
#                     An error occurred while using ast.literal_eval to parse the returned object: {err_msg}
                    
#                     Continue to modify the format of the object so that it meets the requirements of the above format and can be correctly parsed by ast.literal_eval.
#                     Note that you only need to provide the corrected result and mark it with three # symbols formatted like ###reutrned object###.
                    
#                     Typical errors that may occur:
#                     1.Pay attention to whether each element in the tuple has double quotes; if so, they must be removed.
#                     2.Instead of using colons to separate entities and types, bracketed tags are incorrectly used
#                     3.Each tuple can only have 3 elements. The entity type and entity must be in the same element of the tuple (i.e., within the same string) and separated by a colon. It must conform to the format ("subject entity type:subject entity", "relationship content", "object entity type:object entity"). If the type and entity are found in different strings, they need to be combined into one.   
#                     4.The returned object may not be marked with ###
#                     5.Entities may miss type annotations, you need to label their types as required above, while keeping the subject-verb-object order unchanged."""
                
#                 try:
#                     refined_resp2 = refine_entities(format_error_prompt, context)
#                 except:
#                     print("resp parsing error")
#                     continue
#                 print(f"error: {err_msg}, retry")
#                 if max_try==0: print("exceed max retry, continue")

#     logger.info(f"triples:\n {triples}")    

#     return triples

# #all_entities: {doc_path: {entity_tag: [entitiy, ], }, } 
# def extract_KB_triples(entity_list_path, entity_schema_path, chunk_size=200):
#     all_entities = load_dict_from_file(entity_list_path)
#     entity_schema = load_dict_from_file(entity_schema_path)
#     #print(all_entities)
    
#     doc_num = len(all_entities)
#     processed_doc = 0
    
#     all_triples = {}

#     for doc_path in all_entities.keys():
#         print(f"{processed_doc+1}/{doc_num}: {doc_path}")
#         processed_doc += 1

#         entity_list = all_entities[doc_path]
#         if entity_list == {} : continue
#         doc_triples = extract_triple(entity_schema, entity_list, doc_path, chunk_size=chunk_size)
#         all_triples[doc_path] = doc_triples
#         #break
#     return all_triples  
import os
import re
import ast
import logging
# 如果环境里没有 jieba，可以注释掉相关行，改用简单切分
try:
    import jieba
except ImportError:
    jieba = None

# ================= 1. 优化后的文本切分 =================
def split_text(content, chunk_size, lang="chn"):
    """
    更稳健的切分函数，支持中文句号、换行符等。
    """
    if not content: return []
    
    # 预处理：将常见的句子分隔符统一替换为换行符，方便统一处理
    # 保护浮点数（如 3.14）不被切分有点复杂，这里简化处理：
    # 先按换行符切分，再按句号/分号切分
    content = content.replace("。", "\n").replace("；", "\n").replace(";", "\n")
    
    # 按换行符粗切
    lines = content.split('\n')
    lines = [line.strip() for line in lines if line.strip()]
    
    output_chunks = []
    current_chunk = []
    current_tokens = 0

    for line in lines:
        # 简单估算 token 数 (中文字符数 + 英文单词数)
        # 如果没有 jieba，直接用 len(line) 估算
        if lang == "chn" and jieba:
            tokens = len(jieba.lcut(line))
        else:
            tokens = len(line) 
            
        if current_tokens + tokens <= chunk_size:
            current_chunk.append(line)
            current_tokens += tokens
        else:
            if current_chunk:
                output_chunks.append('\n'.join(current_chunk))
            current_chunk = [line]
            current_tokens = tokens

    if current_chunk:
        output_chunks.append('\n'.join(current_chunk))
    
    return output_chunks

# ================= 2. 强力三元组提取正则 =================
def robust_parse_triples(response_str):
    """
    解析三元组列表。
    策略 1: 尝试标准 Python List 解析 (ast.literal_eval)
    策略 2: 如果解析失败 (如截断)，使用正则提取所有形如 ("A", "B", "C") 的元组
    """
    if not response_str: return []
    
    clean_str = response_str.strip()
    # 尝试提取 ### 之间的内容
    if "###" in clean_str:
        try:
            clean_str = clean_str.split("###")[1].strip()
        except:
            pass # 提取失败就用原字符串
            
    # 清理常见的 Markdown 标记
    clean_str = clean_str.replace("```json", "").replace("```python", "").replace("```", "")
    
    triples = []
    
    # --- 策略 1: 整体解析 ---
    try:
        # 预处理：将中文标点替换为英文，防止 ast 报错
        safe_str = clean_str.replace("，", ",").replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
        parsed = ast.literal_eval(safe_str)
        if isinstance(parsed, list):
            # 简单的格式校验
            valid_list = []
            for item in parsed:
                if isinstance(item, (list, tuple)) and len(item) == 3:
                    valid_list.append(tuple(item))
            return valid_list
    except:
        pass # 继续尝试策略 2

    # --- 策略 2: 正则暴力提取 (针对截断情况) ---
    # 匹配模式： ("...", "...", "...") 或者 ["...", "...", "..."]
    # 允许包含中文，允许周围有空白
    try:
        # 这是一个比较通用的正则，匹配括号内的三个引号字符串
        # [(\[] 匹配 ( 或 [
        # \s*["'](.*?)["'] 匹配 "实体" (非贪婪)
        pattern = r'[(\[]\s*["\'](.*?)["\']\s*,\s*["\'](.*?)["\']\s*,\s*["\'](.*?)["\']\s*[)\]]'
        matches = re.findall(pattern, clean_str)
        
        for match in matches:
            # match 是一个 tuple ('Subject', 'Relation', 'Object')
            if len(match) == 3:
                triples.append(match)
        
        if triples:
            logger.info(f"Regex salvaged {len(triples)} triples from malformed response.")
            return triples
            
    except Exception as e:
        logger.error(f"Regex extraction failed: {e}")

    return []

# ================= 3. 实体过滤函数 (保持不变) =================
def filter_entity_by_text(entity_list, text, with_tag=False, entity_schema={}):
    result = []
    # 展平 schema
    flat_schema = {}
    if isinstance(entity_schema, list):
        for item in entity_schema:
            flat_schema.update(item)
    elif isinstance(entity_schema, dict):
        flat_schema = entity_schema

    text_lower = text.lower()
    
    for key, value_list in entity_list.items():
        if not value_list: continue
        
        found_strings = set()
        for string in value_list:
            if not string: continue
            # 简单匹配
            if string in text or string.lower() in text_lower:
                found_strings.add(string)
        
        if found_strings:
            if with_tag:
                for entity in found_strings:
                    comment = flat_schema.get(key, "")
                    result.append(f"{key}: {entity}") # 简化格式供 Prompt 使用
            else:
                result.extend(list(found_strings))
                
    return list(set(result))

# ================= 4. 修复后的提取主函数 =================
def extract_triple(entity_schema, entity_list, doc_path, chunk_size=300): # 稍微加大 chunk_size
    
    # 读取文件
    try:
        with open(doc_path, "r", encoding='utf-8') as f:
            doc_content = f.read()
    except:
        logger.error(f"Read file error: {doc_path}")
        return []

    doc_path = os.path.abspath(doc_path)
    all_doc_triples = []
    
    # 使用新切分逻辑
    chunks = split_text(doc_content, chunk_size, lang="chn")
    
    for idx, text_chunk in enumerate(chunks):   
        logger.info(f"Processing {os.path.basename(doc_path)} | Chunk {idx+1}/{len(chunks)}")
        
        # 过滤当前 chunk 存在的实体
        existing_entities = filter_entity_by_text(entity_list, text_chunk, False)
        
        # 如果实体太少，可能无法构建三元组，跳过以节省时间
        if len(existing_entities) < 2: 
            continue
            
        # 构造 Prompt
        extraction_prompt = f"""
        ----------------Task-------------------
        Identify relationship triples from the provided text based on the provided entity list.
        
        ----------------Input-------------------
        [Text Segment]:
        {text_chunk}
        
        [Candidate Entities] (Only use these entities):
        {existing_entities}
        
        ----------------Requirements-------------------
        1. Output format: A Python List of Tuples.
           Example: [("EntityA", "Relation", "EntityB"), ("EntityC", "Relation", "EntityD")]
        2. Format details: 
           - Use DOUBLE QUOTES for strings.
           - Strictly (Subject, Relation, Object).
        3. Rules:
           - Both Subject and Object MUST be from the [Candidate Entities] list provided above.
           - The Relation should be concise and meaningful.
           - If no relationship exists, return an empty list [].
        
        ----------------Response-------------------
        Return only the list marked with ###.
        ###
        [("Subject", "Relation", "Object")]
        ###
        """
        
        try:
            # 请求模型
            resp = request_model(extraction_prompt)
            
            # 使用强力解析
            triples = robust_parse_triples(resp)
            
            if triples:
                print(f"  -> Chunk {idx+1} found {len(triples)} triples")
                all_doc_triples.extend(triples)
            else:
                # logger.info(f"  -> Chunk {idx+1} found 0 triples")
                pass
                
        except Exception as e:
            logger.error(f"Error processing chunk {idx} of {doc_path}: {e}")
            continue

    # 去重
    unique_triples = list(set(all_doc_triples))
    logger.info(f"Finished {os.path.basename(doc_path)}. Total unique triples: {len(unique_triples)}")
    
    return unique_triples

# ================= 5. 批量入口函数 =================
def extract_KB_triples(entity_list_path, entity_schema_path, chunk_size=300):
    all_entities = load_dict_from_file(entity_list_path)
    entity_schema = load_dict_from_file(entity_schema_path)
    
    if not all_entities:
        logger.error("Entity list is empty!")
        return {}
    
    doc_num = len(all_entities)
    processed_doc = 0
    all_triples = {}

    for doc_path in all_entities.keys():
        print(f"Processing File {processed_doc+1}/{doc_num}: {os.path.basename(doc_path)}")
        
        try:
            entity_list = all_entities[doc_path]
            # 如果该文档没有提取到任何实体，直接跳过三元组提取
            if not entity_list:
                print("  -> No entities found in this doc, skipping.")
                continue
                
            doc_triples = extract_triple(entity_schema, entity_list, doc_path, chunk_size=chunk_size)
            all_triples[doc_path] = doc_triples
            
        except Exception as e:
            logger.error(f"CRITICAL ERROR on file {doc_path}: {e}")
            import traceback
            traceback.print_exc()
            
        processed_doc += 1

    return all_triples

In [ ]:
CHUNK_SIZE = 200
all_triples = extract_KB_triples(f"{OUTPUT_PATH}/all_entities.json", f"{OUTPUT_PATH}/full_entity_schema.json", chunk_size=CHUNK_SIZE)
save_dict_to_file(all_triples, f"{OUTPUT_PATH}/all_triples.json")

In [ ]:
# @parse_model_resp(resp_type="str")
# def eval_triples(prompt, context, model):
#     return request_model(prompt, context, model=model)

# #batch_size: 一次判断几个triple
# def eval_wo_GT(triples_path, batch_size=10):
#     all_triples = load_dict_from_file(triples_path)
    
#     true_triples = 0
#     evaled_triples = 0
#     doc_num = len(all_triples)
#     processed_doc = 0
        
#     for doc_path, doc_triples in all_triples.items():
#         doc_true_triples = 0
#         doc_num_triples = len(doc_triples)
#         if not doc_triples: continue
#         print(f"{processed_doc+1}/{doc_num}: {doc_path}")
#         logger.info(f"{processed_doc+1}/{doc_num}: {doc_path}")
#         with open(doc_path, "r") as f:
#             doc_content = f.read().replace('"', " ").replace('“', " ").replace('”', " ")

#         doc_triples = [doc_triples[i:i + batch_size] for i in range(0, len(doc_triples), batch_size)]
#         for idx, chunk in enumerate(doc_triples):
#             print(f"chunk: {idx+1} / {len(doc_triples)}")
#             logger.info(f"chunk: {idx+1} / {len(doc_triples)}")
#             eval_prompt = f"""
#                 Given a text and up to {batch_size} triples. Each triple consists of (subject, predicate, object). A triple is considered correct if the relationship it represents aligns with the original text.
#                 Analyze the correctness of each triple based on the provided text, count the number of valid triples, and return a number marked with ### before and after it (e.g., ###5###) to indicate the count.
                
#                 Text:
#                 {{
#                 {doc_content}
#                 }}

#                 Triples:
#                 {{
#                 {chunk} 
#                 }}
#                 """
#             context = []
#             max_try = 10
#             first_try = True 
#             while max_try>0:
#                 try:
#                     if first_try:
#                         first_try = False 
#                         refined_resp = eval_triples(eval_prompt, context, model="deepseek/deepseek-r1-distill-llama-70b").strip()
#                     assert (refined_resp.isdigit() and int(refined_resp)<=batch_size)
#                     true_triples += int(refined_resp)
#                     doc_true_triples += int(refined_resp)
#                     evaled_triples += len(chunk)
#                     break
#                 except Exception as err:
#                     err_msg = f"err: {err}"
#                     logger.error(err_msg)
#                     format_error_prompt = f"""
#                         An error occurred while parsing the returned content using ast.literal_eval: {err_msg}.
#                         Continue adjusting the format of the returned object to meet the specified requirements, ensuring the correct number of tuples are marked with ### before and after.
#                         Note: Provide only the corrected result, without explanations or code."""
#                     refined_resp = eval_triples(format_error_prompt, context, model="deepseek/deepseek-r1-distill-llama-70b").strip()
#                     print(f"error: {err_msg}, retry")
#                     max_try -= 1 
#                     if max_try==0: print("exceed max retry, continue")
        
#         processed_doc += 1                
#         logger.info(f"doc true_triples: {doc_true_triples}/{doc_num_triples}")
#         print(f"doc true_triples: {doc_true_triples}/{doc_num_triples}")
        
#     precision = true_triples/evaled_triples
#     recall_number = true_triples
#     avg_recall_num = recall_number/processed_doc
#     return {"precision": precision, "recall_number": recall_number, "avg_recall_num": avg_recall_num}


# 1. 【核心修改】去掉装饰器，获取原始文本
# @parse_model_resp(resp_type="str") <--- 删掉这行
def eval_triples(prompt, context, model):
    return request_model(prompt, context, model=model)

def eval_wo_GT(triples_path, batch_size=10):
    all_triples = load_dict_from_file(triples_path)
    
    true_triples = 0
    evaled_triples = 0
    doc_num = len(all_triples)
    processed_doc = 0
        
    for doc_path, doc_triples in all_triples.items():
        doc_true_triples = 0
        doc_num_triples = len(doc_triples)
        if not doc_triples: continue
        
        print(f"{processed_doc+1}/{doc_num}: {doc_path}")
        logger.info(f"{processed_doc+1}/{doc_num}: {doc_path}")
        
        # 读取文件
        try:
            with open(doc_path, "r", encoding='utf-8') as f:
                doc_content = f.read().replace('"', " ").replace('“', " ").replace('”', " ")
        except Exception as e:
            logger.error(f"Read file error: {e}")
            doc_content = ""

        doc_triples_chunks = [doc_triples[i:i + batch_size] for i in range(0, len(doc_triples), batch_size)]
        
        for idx, chunk in enumerate(doc_triples_chunks):
            print(f"chunk: {idx+1} / {len(doc_triples_chunks)}")
            logger.info(f"chunk: {idx+1} / {len(doc_triples_chunks)}")
            
            base_prompt = f"""
                Given a text and up to {batch_size} triples. Each triple consists of (subject, predicate, object). A triple is considered correct if the relationship it represents aligns with the original text.
                Analyze the correctness of each triple based on the provided text, count the number of valid triples, and return a number marked with ### before and after it (e.g., ###5###) to indicate the count.
                
                Text:
                {{
                {doc_content[:15000]} 
                }}

                Triples:
                {{
                {chunk} 
                }}
                """
            
            context = []
            max_try = 5 # 重试次数不用太多
            
            # 当前使用的 prompt，初始为 base_prompt
            current_prompt = base_prompt
            
            while max_try > 0:
                try:
                    # 2. 获取原始回复
                    raw_resp = eval_triples(current_prompt, context, model="deepseek/deepseek-r1-distill-llama-70b")
                    
                    if not raw_resp:
                        raise ValueError("Empty response")

                    # 3. 【核心修改】使用正则提取 ### 数字 ###
                    # 兼容: ###5###, ### 5 ###, 甚至 text... ### 5 ### ...text
                    match = re.search(r'###\s*(\d+)\s*###', raw_resp)
                    
                    final_num = None
                    if match:
                        final_num = int(match.group(1))
                    else:
                        # 兜底：如果没找到 ###，尝试在整个文本里找单纯的数字（如果只有一个数字的话）
                        # 或者你可以选择在这里抛出异常强迫重试
                        # digits = re.findall(r'\d+', raw_resp)
                        # if len(digits) == 1: final_num = int(digits[0])
                        raise ValueError(f"No ### format found in response: {raw_resp[:50]}...")

                    # 4. 校验数值合理性
                    if final_num is not None and final_num <= len(chunk):
                        true_triples += final_num
                        doc_true_triples += final_num
                        evaled_triples += len(chunk)
                        logger.info(f"Chunk Success: {final_num}")
                        break # 成功，跳出重试循环
                    else:
                        raise ValueError(f"Number {final_num} exceeds batch size {len(chunk)}")

                except Exception as err:
                    err_msg = f"Error: {err}"
                    logger.error(err_msg)
                    print(f"Retry {6-max_try}/5: {err_msg}")
                    
                    # 构造报错后的修正 Prompt
                    # 注意：这里我们更新 prompt，下一次循环会用这个 prompt 请求
                    current_prompt = f"""
                        The previous answer format was incorrect. Error: {err_msg}.
                        Please strict follow the requirement: Return ONLY a number marked with ### before and after (e.g., ###5###).
                        Do not include any code or explanation.
                        """
                    max_try -= 1 
                    if max_try == 0:
                        print("Exceed max retry, skipping this chunk.")
        
        processed_doc += 1                
        logger.info(f"doc true_triples: {doc_true_triples}/{doc_num_triples}")
        print(f"doc true_triples: {doc_true_triples}/{doc_num_triples}")
        
    if evaled_triples == 0: return {"precision": 0, "recall_number": 0, "avg_recall_num": 0}

    precision = true_triples / evaled_triples
    recall_number = true_triples
    avg_recall_num = recall_number / processed_doc
    return {"precision": precision, "recall_number": recall_number, "avg_recall_num": avg_recall_num}

In [ ]:
eval_res =  eval_wo_GT(f"{OUTPUT_PATH}/all_triples.json", 10)

In [ ]:
eval_res

In [ ]:
import pickle

with open("output/summary_faiss_index.bin", "rb") as f:
    obj = pickle.load(f)

print(type(obj))
print(len(obj))
for i, item in enumerate(obj):
    print(f"obj[{i}] ->", type(item))
index, texts = obj
print(index)
print("向量维度 d =", index.d)
print("向量数量 ntotal =", index.ntotal)
print("texts 数量 =", len(texts))
for i in range(3):
    print(f"[{i}] {texts[i][:100]}...")




